# LLM Query Builder

In [1]:
# Select GPU (physical GPU 1 -> visible as cuda:0 inside this process)
%env CUDA_VISIBLE_DEVICES=1

# cap GPU memory usage for this kernel (good citizen on shared GPUs)
import torch
if torch.cuda.is_available():
    torch.cuda.set_per_process_memory_fraction(0.7, device=0)
    print("CUDA is available. Using visible GPU 0 with 70% memory cap.")
else:
    print("CUDA not available. Running on CPU.")


env: CUDA_VISIBLE_DEVICES=1
CUDA is available. Using visible GPU 0 with 70% memory cap.


In [7]:
import json
from typing import Any, Dict
from pathlib import Path

from jsonschema import validate
from jsonschema.exceptions import ValidationError

from transformers import AutoTokenizer, AutoModelForCausalLM
from typing import Any, Dict


In [8]:
# ============================================================
# Domain constants + snapping helpers
# ============================================================
SOFT_TRAIN_FRACTION_RANGE = (0.2, 1.0)
SOFT_CORRUPTION_LEVELS = ["none", "mild", "strong"]
SOFT_REVIEW_BUDGETS = [i / 1000 for i in range(5, 501, 5)]  # 0.005..0.500 step 0.005

def q01(x: float, lo: float, hi: float) -> float:
    """Quantize to 1% steps and clip to [lo, hi]."""
    x = max(lo, min(hi, float(x)))
    return round(x * 100.0) / 100.0

def snap_budget(v: float) -> float:
    """Snap to nearest allowed review budget."""
    v = float(v)
    return float(min(SOFT_REVIEW_BUDGETS, key=lambda b: abs(float(b) - v)))

print("Soft domain ready:",
      "train_fraction", SOFT_TRAIN_FRACTION_RANGE,
      "| budgets", (min(SOFT_REVIEW_BUDGETS), max(SOFT_REVIEW_BUDGETS), "n=", len(SOFT_REVIEW_BUDGETS)),
      "| corruption", SOFT_CORRUPTION_LEVELS)


Soft domain ready: train_fraction (0.2, 1.0) | budgets (0.005, 0.5, 'n=', 100) | corruption ['none', 'mild', 'strong']


### Loading QWEN 3B from transformers

In [9]:
# ============================================================
# Load model/tokenizer
# ============================================================
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
)
model.eval()

print("Loaded:", MODEL_ID)
print("device:", next(model.parameters()).device)


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Loaded: Qwen/Qwen2.5-3B-Instruct
device: cuda:0


## Part 1 structuring user's input

In [10]:
# ============================================================
# JSON extractor
# ============================================================
def extract_first_json_object(text: str) -> str:
    """
    Extract the first top-level JSON object {...} from model output.
    Raises ValueError if none found or braces are unbalanced.
    """
    text = text.strip()
    start = text.find("{")
    if start == -1:
        raise ValueError("No '{' found in model output.")

    depth = 0
    in_str = False
    esc = False

    for i in range(start, len(text)):
        ch = text[i]
        if in_str:
            if esc:
                esc = False
            elif ch == "\\":
                esc = True
            elif ch == '"':
                in_str = False
            continue
        else:
            if ch == '"':
                in_str = True
                continue
            if ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    return text[start:i+1]

    raise ValueError("Unbalanced JSON braces in model output.")


In [11]:
# ============================================================
# Output schema (strict contract for the LLM)
# ============================================================

QUERY_SCHEMA = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "type": "object",
    "additionalProperties": False,
    "required": ["objective", "soft_constraints", "selection"],
    "properties": {
        # ---------------- objective ----------------
        "objective": {
            "type": "object",
            "additionalProperties": False,
            "required": ["type"],
            "properties": {
                "type": {"type": "string", "enum": ["target_min", "delta_improve", "target_band"]},
                "target_min": {"type": "number", "minimum": 0.0, "maximum": 1.0},
                "delta": {"type": "number", "minimum": 0.0, "maximum": 1.0},

                # 2-element tuple [L, U]
                "target_band": {
                    "type": "array",
                    "minItems": 2,
                    "maxItems": 2,
                    "prefixItems": [
                        {"type": "number", "minimum": 0.0, "maximum": 1.0},
                        {"type": "number", "minimum": 0.0, "maximum": 1.0},
                    ],
                    "items": False,
                },
            },
            "allOf": [
                {"if": {"properties": {"type": {"const": "target_min"}}},
                 "then": {"required": ["target_min"]}},
                {"if": {"properties": {"type": {"const": "delta_improve"}}},
                 "then": {"required": ["delta"]}},
                {"if": {"properties": {"type": {"const": "target_band"}}},
                 "then": {"required": ["target_band"]}},
            ],
        },

        # ---------------- soft constraints ----------------
        "soft_constraints": {
            "type": "object",
            "additionalProperties": False,
            "required": [
                "params_soft_train_fraction",
                "params_soft_review_budget",
                "params_soft_corruption_level",
            ],
            "properties": {
                "params_soft_train_fraction": {
                    "type": "object",
                    "additionalProperties": False,
                    "required": ["mode"],
                    "properties": {
                        "mode": {"type": "string", "enum": ["free", "fixed", "range"]},
                        "value": {"type": "number"},
                        # 2-element tuple [lo, hi]
                        "bounds": {
                            "type": "array",
                            "minItems": 2,
                            "maxItems": 2,
                            "prefixItems": [{"type": "number"}, {"type": "number"}],
                            "items": False,
                        },
                    },
                },

                "params_soft_review_budget": {
                    "type": "object",
                    "additionalProperties": False,
                    "required": ["mode"],
                    "properties": {
                        "mode": {"type": "string", "enum": ["free", "fixed", "range"]},
                        "value": {"type": "number"},
                        # 2-element tuple [lo, hi]
                        "bounds": {
                            "type": "array",
                            "minItems": 2,
                            "maxItems": 2,
                            "prefixItems": [{"type": "number"}, {"type": "number"}],
                            "items": False,
                        },
                    },
                },

                "params_soft_corruption_level": {
                    "type": "object",
                    "additionalProperties": False,
                    "required": ["mode"],
                    "properties": {
                        "mode": {"type": "string", "enum": ["free", "fixed", "allowed"]},
                        "value": {"type": "string", "enum": SOFT_CORRUPTION_LEVELS},
                        "allowed": {
                            "type": "array",
                            "minItems": 1,
                            "items": {"type": "string", "enum": SOFT_CORRUPTION_LEVELS},
                        },
                    },
                },
            },
        },

        # ---------------- selection ----------------
        "selection": {
            "type": "object",
            "additionalProperties": False,
            "required": ["k", "mode"],
            "properties": {
                "k": {"type": "integer", "enum": [3, 5, 8]},
                "mode": {"type": "string", "enum": ["best", "cheapest", "balanced"]},
            },
        },
    },
}

In [12]:
# ============================================================
# Defaults + fallback + repair
# ============================================================

DEFAULT_EPS = 0.01
DEFAULT_K = 5
DEFAULT_MODE = "balanced"
MAX_RETRIES = 1

def fallback_query_core(x_factual: Dict[str, Any], y_factual_sur: float,
                        eps: float = DEFAULT_EPS,
                        k: int = DEFAULT_K,
                        mode: str = DEFAULT_MODE) -> Dict[str, Any]:
    """Stable fallback based on factual config (safest)."""
    L = max(0.0, float(y_factual_sur) - float(eps))
    U = min(1.0, float(y_factual_sur) + float(eps))
    if U < L:
        L, U = U, L

    tf = float(x_factual["params_soft_train_fraction"])
    rb = float(snap_budget(x_factual["params_soft_review_budget"]))
    cl = str(x_factual["params_soft_corruption_level"])

    return {
        "objective": {"type": "target_band", "target_band": [L, U]},
        "soft_constraints": {
            "params_soft_train_fraction": {"mode": "fixed", "value": q01(tf, *SOFT_TRAIN_FRACTION_RANGE)},
            "params_soft_review_budget": {"mode": "fixed", "value": rb},
            "params_soft_corruption_level": {"mode": "fixed", "value": cl if cl in SOFT_CORRUPTION_LEVELS else "none"},
        },
        "selection": {"k": int(k), "mode": str(mode)},
    }

In [13]:
# ============================================================
# Contract prompt
# ============================================================
CONTRACT_PROMPT = f"""
Return ONLY valid JSON. No markdown. No extra text.

Keys required: objective, soft_constraints, selection.

objective.type in: target_min | delta_improve | target_band

soft_constraints must contain:
- params_soft_train_fraction: mode free|fixed|range
- params_soft_review_budget: mode free|fixed|range
- params_soft_corruption_level: mode free|fixed|allowed

Allowed domains:
- params_soft_train_fraction in [{SOFT_TRAIN_FRACTION_RANGE[0]}, {SOFT_TRAIN_FRACTION_RANGE[1]}]
- params_soft_review_budget in [{min(SOFT_REVIEW_BUDGETS)}, {max(SOFT_REVIEW_BUDGETS)}] (grid step 0.005)
- params_soft_corruption_level in {SOFT_CORRUPTION_LEVELS}

selection:
- k in [3,5,8]
- mode in ["best","cheapest","balanced"]
""".strip()


In [14]:
# ============================================================
# generation function
# ============================================================
def _clamp(x: float, lo: float, hi: float) -> float:
    return max(lo, min(hi, float(x)))

def repair_and_snap(draft: Dict[str, Any], x_factual: Dict[str, Any]) -> Dict[str, Any]:
    """
    Enforce:
    - objective numeric sanity + L<=U
    - soft_train_fraction: clamp + q01() if fixed; bounds if range
    - soft_review_budget: snap_budget() if fixed; snap bounds if range
    - soft_corruption_level: filter allowed; fallback to factual if empty/invalid
    """
    d = json.loads(json.dumps(draft))  # deep copy

    # ---------------- objective ----------------
    obj = d["objective"]
    t = obj["type"]

    if t == "target_min":
        obj["target_min"] = _clamp(obj["target_min"], 0.0, 1.0)
    elif t == "delta_improve":
        obj["delta"] = _clamp(obj["delta"], 0.0, 1.0)
    elif t == "target_band":
        L, U = obj["target_band"]
        L = _clamp(L, 0.0, 1.0)
        U = _clamp(U, 0.0, 1.0)
        if U < L:
            L, U = U, L
        obj["target_band"] = [L, U]
    else:
        # should never happen if schema validated, but keep safe
        obj.clear()
        obj.update({"type": "target_band", "target_band": [0.0, 1.0]})

    # -------------- soft constraints --------------
    sc = d["soft_constraints"]

    # (1) train fraction
    tf = sc["params_soft_train_fraction"]
    lo_tf, hi_tf = SOFT_TRAIN_FRACTION_RANGE
    if tf["mode"] == "fixed":
        tf["value"] = q01(float(tf["value"]), lo_tf, hi_tf)
    elif tf["mode"] == "range":
        lo, hi = tf["bounds"]
        lo = q01(float(lo), lo_tf, hi_tf)
        hi = q01(float(hi), lo_tf, hi_tf)
        if hi < lo:
            lo, hi = hi, lo
        tf["bounds"] = [lo, hi]
    # free -> nothing

    # (2) review budget
    rb = sc["params_soft_review_budget"]
    rb_min, rb_max = float(min(SOFT_REVIEW_BUDGETS)), float(max(SOFT_REVIEW_BUDGETS))
    if rb["mode"] == "fixed":
        rb["value"] = float(snap_budget(_clamp(rb["value"], rb_min, rb_max)))
    elif rb["mode"] == "range":
        lo, hi = rb["bounds"]
        lo = float(snap_budget(_clamp(lo, rb_min, rb_max)))
        hi = float(snap_budget(_clamp(hi, rb_min, rb_max)))
        if hi < lo:
            lo, hi = hi, lo
        rb["bounds"] = [lo, hi]
    # free -> nothing

    # (3) corruption level
    cl = sc["params_soft_corruption_level"]
    allowed_set = set(SOFT_CORRUPTION_LEVELS)

    if cl["mode"] == "fixed":
        v = str(cl["value"])
        if v not in allowed_set:
            cl["value"] = str(x_factual["params_soft_corruption_level"])
            if cl["value"] not in allowed_set:
                cl["value"] = "none"

    elif cl["mode"] == "allowed":
        vals = [str(v) for v in cl.get("allowed", [])]
        vals = [v for v in vals if v in allowed_set]
        vals = sorted(set(vals))
        if not vals:
            # safest: fix to factual if allowed becomes empty
            factual_v = str(x_factual["params_soft_corruption_level"])
            if factual_v not in allowed_set:
                factual_v = "none"
            cl.clear()
            cl.update({"mode": "fixed", "value": factual_v})
        else:
            cl["allowed"] = vals

    # free -> nothing

    # ---------------- selection ----------------
    sel = d["selection"]
    # schema already restricts k/mode, but keep safe types:
    sel["k"] = int(sel["k"])
    sel["mode"] = str(sel["mode"])

    return d
@torch.inference_mode()
def llama_generate_json(user_text: str, max_new_tokens: int = 400) -> str:
    if hasattr(tokenizer, "apply_chat_template"):
        messages = [
            {"role": "system", "content": "Output ONLY valid JSON. No extra text."},
            {"role": "user", "content": user_text},
        ]
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    else:
        prompt = "SYSTEM: Output ONLY valid JSON.\nUSER:\n" + user_text + "\nASSISTANT:\n"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=0.0,
        top_p=1.0,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    decoded = tokenizer.decode(out[0], skip_special_tokens=True)
    return extract_first_json_object(decoded)


In [15]:
# ============================================================
# Normalization layer: coerce model "shorthand" into canonical schema
# ============================================================

REQUIRED_SOFT_KEYS = {
    "params_soft_train_fraction",
    "params_soft_review_budget",
    "params_soft_corruption_level",
}

def normalize_objective(obj: Dict[str, Any]) -> Dict[str, Any]:
    """
    Accepts common variants:
    - {"type":"target_min","value":0.8}  -> {"type":"target_min","target_min":0.8}
    - {"type":"delta_improve","value":0.03} -> {"type":"delta_improve","delta":0.03}
    - {"type":"target_band","value":[L,U]}  -> {"type":"target_band","target_band":[L,U]}
    """
    if not isinstance(obj, dict):
        raise ValueError("objective must be an object/dict")

    t = obj.get("type")
    if t == "target_min" and "target_min" not in obj and "value" in obj:
        obj = dict(obj)
        obj["target_min"] = obj.pop("value")
        return obj

    if t == "delta_improve" and "delta" not in obj and "value" in obj:
        obj = dict(obj)
        obj["delta"] = obj.pop("value")
        return obj

    if t == "target_band" and "target_band" not in obj and "value" in obj:
        obj = dict(obj)
        obj["target_band"] = obj.pop("value")
        return obj

    return obj


def normalize_soft_constraints(sc: Dict[str, Any]) -> Dict[str, Any]:
    """
    Handles:
    - canonical format
    - wrapper {mode, values/value}
    - flat values
    - shorthand format (as produced by your model right now)
    """
    if not isinstance(sc, dict):
        raise ValueError("soft_constraints must be an object/dict")

    # --- If already canonical, strip extras ---
    if REQUIRED_SOFT_KEYS.issubset(sc.keys()) and all(isinstance(sc[k], dict) for k in REQUIRED_SOFT_KEYS):
        return {k: sc[k] for k in REQUIRED_SOFT_KEYS}

    # --- Wrapper {"mode":..., "values"/"value": {...}} ---
    if "mode" in sc and ("values" in sc or "value" in sc):
        mode = sc.get("mode")
        vals = sc.get("values", sc.get("value"))
        if isinstance(vals, dict):
            # canonical inside wrapper?
            if REQUIRED_SOFT_KEYS.issubset(vals.keys()):
                # could still be shorthand inside, recurse
                return normalize_soft_constraints(vals)
            # else: fall through to treat as flat dict
            sc = vals
        else:
            # values is not dict (could be list for corruption) -> fall through to shorthand parsing
            pass

    # --- Shorthand parsing (key symptom: params_* keys are strings like "fixed"/"range"/"allowed") ---
    out = {}

    # 1) train_fraction
    tf_mode = sc.get("params_soft_train_fraction")
    if isinstance(tf_mode, str) and tf_mode in {"free", "fixed", "range"}:
        if tf_mode == "free":
            out["params_soft_train_fraction"] = {"mode": "free"}
        elif tf_mode == "fixed":
            # try tf value from sc["train_fraction"] or sc["value"] (if user gave only one)
            v = sc.get("train_fraction", sc.get("params_soft_train_fraction_value", sc.get("value")))
            if v is None:
                # leave incomplete -> validation will fail -> retry/fallback
                out["params_soft_train_fraction"] = {"mode": "fixed", "value": 0.2}
            else:
                out["params_soft_train_fraction"] = {"mode": "fixed", "value": v}
        else:  # range
            lo = sc.get("train_fraction_min", sc.get("min", sc.get("lower")))
            hi = sc.get("train_fraction_max", sc.get("max", sc.get("upper")))
            # If the model didn't provide bounds, keep a sensible default band
            if lo is None or hi is None:
                lo, hi = 0.25, 0.55
            out["params_soft_train_fraction"] = {"mode": "range", "bounds": [lo, hi]}
    elif isinstance(sc.get("params_soft_train_fraction"), dict):
        out["params_soft_train_fraction"] = sc["params_soft_train_fraction"]

    # 2) review_budget
    rb_mode = sc.get("params_soft_review_budget")
    if isinstance(rb_mode, str) and rb_mode in {"free", "fixed", "range"}:
        if rb_mode == "free":
            out["params_soft_review_budget"] = {"mode": "free"}
        elif rb_mode == "fixed":
            v = sc.get("review_budget", sc.get("params_soft_review_budget_value", sc.get("value")))
            out["params_soft_review_budget"] = {"mode": "fixed", "value": v}
        else:
            lo = sc.get("review_budget_min", sc.get("min_budget", sc.get("lower_budget")))
            hi = sc.get("review_budget_max", sc.get("max_budget", sc.get("upper_budget")))
            if lo is None or hi is None:
                lo, hi = 0.02, 0.05
            out["params_soft_review_budget"] = {"mode": "range", "bounds": [lo, hi]}
    elif isinstance(sc.get("params_soft_review_budget"), dict):
        out["params_soft_review_budget"] = sc["params_soft_review_budget"]

    # 3) corruption_level
    cl_mode = sc.get("params_soft_corruption_level")
    if isinstance(cl_mode, str) and cl_mode in {"free", "fixed", "allowed"}:
        if cl_mode == "free":
            out["params_soft_corruption_level"] = {"mode": "free"}
        elif cl_mode == "fixed":
            v = sc.get("corruption_level", sc.get("params_soft_corruption_level_value", sc.get("value")))
            out["params_soft_corruption_level"] = {"mode": "fixed", "value": v}
        else:  # allowed
            vals = sc.get("allowed", sc.get("values", sc.get("levels")))
            out["params_soft_corruption_level"] = {"mode": "allowed", "allowed": vals if vals is not None else ["none"]}
    elif isinstance(sc.get("params_soft_corruption_level"), dict):
        out["params_soft_corruption_level"] = sc["params_soft_corruption_level"]

    # --- Flat values fallback: if user gave actual numbers/strings at root ---
    # e.g. {"params_soft_review_budget":0.03,...}
    if not REQUIRED_SOFT_KEYS.issubset(out.keys()):
        flat_hits = [k for k in REQUIRED_SOFT_KEYS if k in sc and not isinstance(sc[k], dict)]
        if flat_hits:
            for k in flat_hits:
                out[k] = {"mode": "fixed", "value": sc[k]}
                
    missing = REQUIRED_SOFT_KEYS - set(out.keys())
    if missing:
        raise ValueError(f"soft_constraints missing keys after normalization: {sorted(missing)}")

    return out

In [16]:
# ============================================================
# Build a safe query core (normalize -> validate -> repair -> validate)
# ============================================================

def build_query_core(user_text: str,
                     x_factual: Dict[str, Any],
                     y_factual_sur: float,
                     max_retries: int = MAX_RETRIES) -> Dict[str, Any]:
    last_error = None

    for attempt in range(max_retries + 1):
        try:
            prompt = CONTRACT_PROMPT + "\n\nUSER REQUEST:\n" + user_text.strip()
            raw = llama_generate_json(prompt)

            draft = json.loads(raw)

            # Normalize objective + soft_constraints BEFORE validation
            if "objective" in draft:
                draft["objective"] = normalize_objective(draft["objective"])
            if "soft_constraints" in draft:
                draft["soft_constraints"] = normalize_soft_constraints(draft["soft_constraints"])

            validate(instance=draft, schema=QUERY_SCHEMA)

            repaired = repair_and_snap(draft, x_factual)
            validate(instance=repaired, schema=QUERY_SCHEMA)

            repaired["_meta"] = {"used_fallback": False, "attempts": attempt + 1}
            return repaired

        except Exception as e:
            last_error = e

    core = fallback_query_core(x_factual, y_factual_sur)
    core["_meta"] = {"used_fallback": True, "reason": str(last_error)[:250]}
    return core


In [17]:
# ============================================================
# Assemble full query
# ============================================================
def assemble_full_query(core: Dict[str, Any],
                        selected_idx: int,
                        x_factual: Dict[str, Any],
                        y_factual_true: float,
                        y_factual_sur: float) -> Dict[str, Any]:
    return {
        "factual": {
            "index": int(selected_idx),
            "x": dict(x_factual),
            "value_true": float(y_factual_true),
            "value_surrogate": float(y_factual_sur),
        },
        "objective": core["objective"],
        "soft_constraints": core["soft_constraints"],
        "selection": core["selection"],
        "_meta": core.get("_meta", {}),
    }


In [18]:
# ============================================================
# End-to-end test
# ============================================================
# Fake factual for testing (must match your real x_factual keys used in CF notebook)
_xf = {
    "params_soft_train_fraction": 0.37,
    "params_soft_review_budget": 0.031,  # off-grid on purpose
    "params_soft_corruption_level": "mild",
}

user_text = (
    "Reach at least 0.80 quality. "
    "Fix review budget to 0.033 (approximately). "
    "Allow corruption none or mild. "
    "Keep train fraction between 0.25 and 0.55. "
    "Return 5 options, balanced."
)

core = build_query_core(user_text=user_text, x_factual=_xf, y_factual_sur=0.72)
print("CORE:\n", json.dumps(core, indent=2))

full = assemble_full_query(core, selected_idx=123, x_factual=_xf, y_factual_true=0.70, y_factual_sur=0.72)
print("\nFULL QUERY:\n", json.dumps(full, indent=2))


The following generation flags are not valid and may be ignored: ['temperature', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


CORE:
 {
  "objective": {
    "type": "target_min",
    "target_min": 0.8
  },
  "soft_constraints": {
    "params_soft_train_fraction": {
      "mode": "range",
      "bounds": [
        0.25,
        0.55
      ]
    },
    "params_soft_review_budget": {
      "mode": "fixed",
      "value": 0.035
    },
    "params_soft_corruption_level": {
      "mode": "allowed",
      "allowed": [
        "mild",
        "none"
      ]
    }
  },
  "selection": {
    "k": 5,
    "mode": "balanced"
  },
  "_meta": {
    "used_fallback": false,
    "attempts": 1
  }
}

FULL QUERY:
 {
  "factual": {
    "index": 123,
    "x": {
      "params_soft_train_fraction": 0.37,
      "params_soft_review_budget": 0.031,
      "params_soft_corruption_level": "mild"
    },
    "value_true": 0.7,
    "value_surrogate": 0.72
  },
  "objective": {
    "type": "target_min",
    "target_min": 0.8
  },
  "soft_constraints": {
    "params_soft_train_fraction": {
      "mode": "range",
      "bounds": [
        0.25,
 

In [55]:
prompt = CONTRACT_PROMPT + "\n\nUSER REQUEST:\n" + user_text.strip()
raw = llama_generate_json(prompt)
print("RAW MODEL OUTPUT:\n", raw[:1500], "...\n")
draft = json.loads(raw)
print("DRAFT soft_constraints keys:", list(draft.get("soft_constraints", {}).keys()))


RAW MODEL OUTPUT:
 {"objective": {"type": "target_min", "value": 0.8}, "soft_constraints": {"params_soft_train_fraction": "range", "mode": "free", "params_soft_review_budget": "fixed", "value": 0.033, "params_soft_corruption_level": "allowed", "values": ["none", "mild"]}, "selection": {"k": 5, "mode": "balanced"}} ...

DRAFT soft_constraints keys: ['params_soft_train_fraction', 'mode', 'params_soft_review_budget', 'value', 'params_soft_corruption_level', 'values']


# Part 2 — Present Counterfactuals (User-friendly report)

Input: `surrogate_pkl_cfs_metadata/cfs.csv`

Outputs:
- ranked table (top-k)
- per-CF “cards” (actionable explanation)
- optional LLM narrative summary


In [63]:

META_PATH = Path("surrogate_pkl_cfs_metadata/metadata.json")
assert META_PATH.is_file(), f"Missing metadata file: {META_PATH.resolve()}"

with open(META_PATH, "r") as f:
    metadata = json.load(f)

print("Metadata keys:", metadata.keys())


Metadata keys: dict_keys(['surrogate_model_path', 'splits_dir', 'target', 'features', 'categorical', 'factual_index', 'factual_value_surrogate', 'objective', 'soft_constraints', 'selection', 'desired_range', 'permitted_range', 'n_cfs'])


In [68]:
FEATURES = metadata["features"]

def to_python_scalar(x):
    """Convert numpy scalars to native Python types."""
    if isinstance(x, (np.integer,)):
        return int(x)
    if isinstance(x, (np.floating,)):
        return float(x)
    if isinstance(x, (np.bool_,)):
        return bool(x)
    return x


def extract_full_config(cf_row, features):
    config = {}
    for f in features:
        if f in cf_row:
            config[f] = to_python_scalar(cf_row[f])
    return config


In [69]:
def build_cf_summary(cf_row, objective, soft_constraints):
    score = float(cf_row[SCORE_COL])

    summary = {
        "score": score,
        "objective": objective,
        "objective_status": objective_status(score, objective),
        "constraints": constraint_summary(cf_row, soft_constraints),
        "soft_values": {
            k: cf_row.get(k)
            for k in soft_constraints.keys()
            if k in cf_row
        }
    }
    return summary


In [70]:
@torch.inference_mode()
def explain_cf_with_llm(summary_dict, max_new_tokens=180):
    prompt = (
        "You are explaining a counterfactual configuration to a technical engineer.\n"
        "Use 4-6 concise bullet points.\n"
        "Do not invent numbers.\n\n"
        f"STRUCTURED INPUT:\n{json.dumps(summary_dict, indent=2)}\n\n"
        "EXPLANATION:\n"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    text = tokenizer.decode(out[0], skip_special_tokens=True)
    return text.split("EXPLANATION:")[-1].strip()


In [71]:
for i in range(len(cfs)):
    cf_row = cfs.iloc[i]

    full_config = extract_full_config(cf_row, FEATURES)
    summary = build_cf_summary(cf_row, objective, soft_constraints)
    explanation = explain_cf_with_llm(summary)

    print("=" * 100)
    print(f"CF #{cf_row.get('cf_id', i)}")
    print("\n[ PERFORMANCE ]")
    print(f"Surrogate score: {summary['score']:.4f}")
    print(summary["objective_status"])

    print("\n[ FULL CONFIGURATION ]")
    print(json.dumps(full_config, indent=2))

    print("\n[ INTERPRETATION ]")
    print(explanation)
    print("\n")


CF #0

[ PERFORMANCE ]
Surrogate score: 0.8945
✓ Meets minimum target (0.8)

[ FULL CONFIGURATION ]
{
  "params_backbone": "resnet34",
  "params_batch_size": 16,
  "params_center_crop_key": 0.875,
  "params_coreset_sampling_ratio": 0.099,
  "params_image_size_key": 224,
  "params_layers_key": "l3",
  "params_max_patches_per_image": 1024,
  "params_num_neighbors": 2,
  "params_pre_trained": true,
  "params_reduction": "mean",
  "params_soft_corruption_level": "none",
  "params_soft_review_budget": 0.365,
  "params_soft_train_fraction": 0.65
}

[ INTERPRETATION ]
The model's score is 0.894, which meets the minimum target of 0.8.
However, there are some constraints that need to be addressed:

- The `params_soft_train_fraction` is set to 0.65, which is outside the allowed range of [0.25, 0.55].
- The `params_soft_review_budget` is set to 0.365, which exceeds the fixed value of 0.035.
- The `params_soft_corruption_level` is at its allowed level of 'none'.

These issues need to be corrected 

In [72]:
for i in range(len(cfs)):
    print("=" * 90)
    print(f"CF #{cfs.iloc[i].get('cf_id', i)}")
    print(explain_cf_context(cfs.iloc[i], objective, soft_constraints))
    print("\n")


CF #0
Surrogate score: 0.8945
✓ Meets minimum target (0.8)

Constraint evaluation:
  - params_soft_train_fraction: ✗ outside allowed range [0.25, 0.55]
  - params_soft_review_budget: ✗ violates fixed value (got 0.365, expected 0.035)
  - params_soft_corruption_level: ✓ allowed (none)

Training effort level: 0.65
Review effort level: 0.365
Robustness setting: none


CF #1
Surrogate score: 0.8908
✓ Meets minimum target (0.8)

Constraint evaluation:
  - params_soft_train_fraction: ✗ outside allowed range [0.25, 0.55]
  - params_soft_review_budget: ✗ violates fixed value (got 0.344, expected 0.035)
  - params_soft_corruption_level: ✓ allowed (none)

Training effort level: 0.89
Review effort level: 0.344
Robustness setting: none


CF #2
Surrogate score: 0.8903
✓ Meets minimum target (0.8)

Constraint evaluation:
  - params_soft_train_fraction: ✗ outside allowed range [0.25, 0.55]
  - params_soft_review_budget: ✗ violates fixed value (got 0.365, expected 0.035)
  - params_soft_corruption_lev